In [1]:
import os
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [7]:
from pathlib import Path

def resolve_repo_root():
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent.parent)
    candidates.append(Path.cwd())
    for candidate in candidates:
        if (candidate / "data" / "processed" / "train.csv").exists():
            return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "processed" / "train.csv").exists():
            return candidate
    return candidates[0] if candidates else Path.cwd()

REPO_ROOT = resolve_repo_root()
TRAIN_CSV = REPO_ROOT / "data" / "processed" / "train.csv"
VAL_CSV = REPO_ROOT / "data" / "processed" / "val.csv"
TEST_CSV = REPO_ROOT / "data" / "processed" / "test.csv"

FEATURE_COLS = [
    "num_vars", "num_assertions", "num_uninterpreted_funcs",
    "num_func_applications", "max_func_nesting_depth",
    "ast_node_count", "max_depth", "num_arith_ops", "file_size_bytes"
]
TARGET_COL = "result"

In [3]:
def load_split(path):
    df = pd.read_csv(path)
    return df[FEATURE_COLS], df[TARGET_COL]

In [4]:
X_train, y_train = load_split(TRAIN_CSV)
X_val, y_val = load_split(VAL_CSV)

print(f"{'max_depth':>10} | {'train_acc':>10} | {'val_acc':>10} | {'gap':>6}")
print("-" * 45)

for depth in [2, 3, 4, 5, 6, 8, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, clf.predict(X_train))
    val_acc = accuracy_score(y_val, clf.predict(X_val))
    gap = train_acc - val_acc

    depth_label = "None" if depth is None else str(depth)
    print(f"{depth_label:>10} | {train_acc:>10.3f} | {val_acc:>10.3f} | {gap:>6.3f}")

 max_depth |  train_acc |    val_acc |    gap
---------------------------------------------
         2 |      0.823 |      0.750 |  0.073
         3 |      0.844 |      0.781 |  0.063
         4 |      0.897 |      0.844 |  0.054
         5 |      0.930 |      0.891 |  0.039
         6 |      0.956 |      0.875 |  0.081
         8 |      0.983 |      0.891 |  0.092
      None |      1.000 |      0.891 |  0.109


Depth 5 seems best due to tied val_acc and gap min, indicating optimal fit to training.

In [5]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# Combine train + val for cross-validation, since k-fold does its own internal splitting
# (test set stays completely untouched, reserved for final evaluation only)
X_cv = pd.concat([X_train, X_val], ignore_index=True)
y_cv = pd.concat([y_train, y_val], ignore_index=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'max_depth':>10} | {'mean_acc':>10} | {'std':>6}")
print("-" * 32)

cv_results = {}
for depth in [2, 3, 4, 5, 6, 8, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(clf, X_cv, y_cv, cv=skf, scoring="accuracy")
    cv_results[depth] = scores
    depth_label = "None" if depth is None else str(depth)
    print(f"{depth_label:>10} | {scores.mean():>10.3f} | {scores.std():>6.3f}")

 max_depth |   mean_acc |    std
--------------------------------
         2 |      0.817 |  0.007
         3 |      0.851 |  0.029
         4 |      0.886 |  0.009
         5 |      0.914 |  0.011
         6 |      0.924 |  0.012
         8 |      0.942 |  0.010
      None |      0.942 |  0.006


Under this more rigorous check, we see accuracy continues growing with depth with stable variance, with depth >5 showing modest accuracy improvements. So unlimited depth (>=8) may be the right choice if we are willing to sacrifice the readability of the tree.

Just for the sake of curiosity, I will try both depth 5 and unlimited depth on the test set.

In [8]:
from sklearn.metrics import accuracy_score, classification_report

X_test, y_test = load_split(TEST_CSV)

candidates = {
    "depth_5": DecisionTreeClassifier(max_depth=5, random_state=42),
    "depth_8": DecisionTreeClassifier(max_depth=8, random_state=42),
    "unlimited": DecisionTreeClassifier(max_depth=None, random_state=42),
}

for name, clf in candidates.items():
    clf.fit(X_cv, y_cv)  # train on combined train+val, same data the k-fold CV used
    test_preds = clf.predict(X_test)
    test_acc = accuracy_score(y_test, test_preds)
    print(f"\n=== {name} ===")
    print(f"Test accuracy: {test_acc:.3f}")
    print(classification_report(y_test, test_preds))


=== depth_5 ===
Test accuracy: 0.855
              precision    recall  f1-score   support

         sat       0.87      0.91      0.89        43
     unknown       0.71      0.71      0.71         7
       unsat       0.88      0.79      0.83        19

    accuracy                           0.86        69
   macro avg       0.82      0.80      0.81        69
weighted avg       0.86      0.86      0.85        69


=== depth_8 ===
Test accuracy: 0.812
              precision    recall  f1-score   support

         sat       0.83      0.88      0.85        43
     unknown       0.67      0.57      0.62         7
       unsat       0.82      0.74      0.78        19

    accuracy                           0.81        69
   macro avg       0.77      0.73      0.75        69
weighted avg       0.81      0.81      0.81        69


=== unlimited ===
Test accuracy: 0.855
              precision    recall  f1-score   support

         sat       0.87      0.91      0.89        43
     unknown 

A depth of 5 actually performed better to unseen data. I knew this all along and was gonna choose depth 5 anyway. The biggest difference in performance is due to recall on the unknown class (0.57/0.71 for depth_8/depth_5).